###  Install library

In [43]:
 !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
 !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Initial Setup

In [45]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [46]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [47]:
llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)

In [48]:
from pathlib import Path

relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break
doc_path = str(Path(*relevant_parts) / "sonnets.txt")

In [49]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

### Adding a second knowledge source

In [50]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://docs.astral.sh/ruff/faq/"
)

documents = loader.load()

### Split the web page

In [51]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)

texts = text_splitter.split_documents(documents)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


In [52]:


ruffsearch = Chroma.from_documents(
    texts,
    embeddings,
    collection_name="ruff-faq"
)

### Create the agent

In [53]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

In [54]:
from langchain.chains import RetrievalQA

sonnets = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever()
)

In [55]:
from langchain.chains import RetrievalQA

ruff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ruffsearch.as_retriever()
)

In [56]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="Useful for when you need to answer questions about Shakespeare's sonnets. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="Useful for when you need to answer questions about Ruff (a Python linter). Input should be a fully formed question.",
    ),
]

In [57]:
from langchain.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

C:\Users\Shrabani P\AppData\Local\Temp\ipykernel_9140\1829271026.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [59]:
agent.invoke(
    "What is the main theme of the sonnets?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 The main theme of the sonnets is a common question, so there should be a specific tool to answer it.
Action: Sonnets QA System
Action Input: "What is the main theme of the sonnets?"
Observation:  The main theme of the sonnets is the power and inevitability of time and mortality, and the struggle to preserve beauty and love in the face of this.
Thought: This answer seems accurate and comprehensive.
Final Answer: The main theme of the sonnets is the power and inevitability of time and mortality, and the struggle to preserve beauty and love in the face of this.

> Finished chain.


{'input': 'What is the main theme of the sonnets?',
 'output': 'The main theme of the sonnets is the power and inevitability of time and mortality, and the struggle to preserve beauty and love in the face of this.'}

In [60]:
agent.invoke(
    "What does the sonnets document say about love?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Sonnets QA System to answer this question.
Action: Sonnets QA System
Action Input: "What does the sonnets document say about love?"
Observation:  The sonnets document explores the complexities and changes of love over time. It suggests that love can grow and strengthen, but also be affected by external factors such as time and societal expectations. The speaker also reflects on the idea of expressing love too openly and losing its specialness.
Thought: This information is helpful, but I should also consider using the Ruff QA System to see if it has any additional insights.
Action: Ruff QA System
Action Input: "What does the sonnets document say about love?"
Observation:  I don't know, as the context provided does not mention a sonnets document.
Thought: It seems like the Sonnets QA System is the better tool for this question.
Final Answer: The Sonnets document explores the complexities and changes of love over time, suggesting that it can grow and strengthen but also 

{'input': 'What does the sonnets document say about love?',
 'output': 'The Sonnets document explores the complexities and changes of love over time, suggesting that it can grow and strengthen but also be affected by external factors. The speaker also reflects on the idea of expressing love too openly and losing its specialness.'}

In [61]:
agent.invoke(
    "What does Shakespeare say about love in the sonnets?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 Shakespeare's sonnets are known for their exploration of love and its complexities.
Action: Sonnets QA System
Action Input: "What does Shakespeare say about love in the sonnets?"
Observation:  Shakespeare explores the complexities and challenges of love in his sonnets, often highlighting the effects of time and change on relationships. He also emphasizes the power and endurance of love, even in the face of obstacles and difficulties.
Thought: This information gives me a good understanding of Shakespeare's perspective on love in his sonnets.
Final Answer: Shakespeare explores the complexities and challenges of love in his sonnets, emphasizing its power and endurance despite obstacles and change.

> Finished chain.


{'input': 'What does Shakespeare say about love in the sonnets?',
 'output': 'Shakespeare explores the complexities and challenges of love in his sonnets, emphasizing its power and endurance despite obstacles and change.'}

## Use the Agent solely as a router

In [62]:
agent.invoke(
    {
        "input": "What does Shakespeare say about love in the sonnets?"
    }
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Sonnets QA System to answer this question.
Action: Sonnets QA System
Action Input: "What does Shakespeare say about love in the sonnets?"
Observation:  Shakespeare explores the complexities and challenges of love in his sonnets, often highlighting the effects of time and change on relationships. He also emphasizes the power and endurance of love, even in the face of obstacles and difficulties.
Thought: This is a good overview, but I should also consider specific sonnets that address love.
Action: Sonnets QA System
Action Input: "Which sonnets specifically address love?"
Observation:  Sonnets 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86
Thought: This is a comprehensive list, but I should a

{'input': 'What does Shakespeare say about love in the sonnets?',
 'output': 'Shakespeare explores the complexities and challenges of love in his sonnets, emphasizing its power and endurance, while also acknowledging its potential for change and the need for effort to sustain it.'}

In [63]:
agent.invoke(
    {
        "input": "What is Ruff used for?"
    }
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 Ruff is a Python linter, so it is likely used for checking code for errors and enforcing coding standards.
Action: Ruff QA System
Action Input: "What is Ruff used for?"
Observation:  Ruff is a Python linter and formatter that can be used to improve code quality and consistency. It can also replace other tools such as Black, isort, and eradicate.
Thought: This confirms my initial thought.
Final Answer: Ruff is used for checking code for errors, enforcing coding standards, and improving code quality and consistency. It can also replace other tools such as Black, isort, and eradicate.

> Finished chain.


{'input': 'What is Ruff used for?',
 'output': 'Ruff is used for checking code for errors, enforcing coding standards, and improving code quality and consistency. It can also replace other tools such as Black, isort, and eradicate.'}